# 01 · La aplicación logística: del orden al caos

Taller «Entre el orden y el caos». Este notebook corre de principio a fin en **Google Colab** o en tu
computadora (Entorno de ejecución → Ejecutar todo). Es determinista: no hay azar, la condición inicial
es fija ($x_0 = 0.4$).

$$x_{n+1} = r\,x_n\,(1 - x_n)$$

Qué vas a obtener:
1. el diagrama de bifurcación;
2. la sensibilidad a condiciones iniciales;
3. el exponente de Lyapunov (dónde empieza el caos);
4. la constante de Feigenbaum $\delta$ en dos mapas distintos (universalidad).

Figuras en `figuras/nb01_*.png` y números en `resultados/nb01_logistica.json`. Los mismos cálculos,
como script, están en `simulaciones/logistica.py`.

In [ ]:
# Configuración: ¿Colab o local? ¿Dónde se guardan figuras y números?
import sys, json, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Cambia esta ruta por la carpeta del kit en tu Drive (Mi unidad/kit-taller).
    RAIZ = Path("/content/drive/MyDrive/kit-taller")
else:
    # Local: el notebook vive en notebooks/, la raíz del repositorio es la carpeta de arriba.
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FIG = RAIZ / "figuras"
RES = RAIZ / "resultados"
FIG.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)
print("Colab:", EN_COLAB, "| figuras en", FIG, "| números en", RES)

## Cálculo en GPU si hay, si no en NumPy

Si el entorno tiene GPU y CuPy (en Colab: Entorno de ejecución → Cambiar tipo → GPU), los barridos en
$r$ corren en la GPU. Si no, el mismo código corre con NumPy. El resultado es el mismo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TINTA, TINTA2, AZUL, NARANJA, AQUA = "#0b0b0b", "#52514e", "#2a78d6", "#eb6834", "#1baf7a"
SECUENCIAL = ["#9cc3ee", "#5f9de4", "#2a78d6", "#1b4f8f"]   # un solo tono, para series ordenadas
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e1", "font.size": 9, "legend.frameon": False})

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIG / nombre, dpi=200)
    print("figura ->", FIG / nombre)

RESULTADOS = {}
def guardar_resultados(nombre):
    (RES / nombre).write_text(json.dumps(RESULTADOS, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
    print("números ->", RES / nombre)

try:
    import cupy as xp
    xp.zeros(1)                      # falla aquí si no hay GPU utilizable
    EN_GPU = True
except Exception:
    xp = np
    EN_GPU = False
print("Cálculo en", "GPU (CuPy)" if EN_GPU else "CPU (NumPy)")

def a_numpy(a):
    return a.get() if EN_GPU else a

X0 = 0.4

## 1. Diagrama de bifurcación

In [ ]:
def atractor(r_min, r_max, n_r=3000, transitorio=1500, muestras=300):
    """Para cada r, itera desde X0, descarta el transitorio y guarda los puntos siguientes."""
    r = xp.linspace(r_min, r_max, n_r)
    x = xp.full(n_r, X0)
    for _ in range(transitorio):
        x = r * x * (1 - x)
    puntos = xp.empty((muestras, n_r))
    for j in range(muestras):
        x = r * x * (1 - x)
        puntos[j] = x
    return a_numpy(r), a_numpy(puntos)

r, puntos = atractor(2.5, 4.0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.tile(r, puntos.shape[0]), puntos.ravel(), ",", color=TINTA, alpha=0.25, rasterized=True)
for rr, t in [(3.0, "periodo 2"), (3.449, "periodo 4"), (3.5699, "caos"), (3.8284, "ventana de periodo 3")]:
    ax.axvline(rr, color=NARANJA, lw=0.7)
    ax.text(rr + 0.006, 0.03, t, rotation=90, fontsize=7, color=TINTA2)
ax.set(xlabel="tasa de crecimiento r", ylabel="población a largo plazo x", xlim=(2.5, 4), ylim=(0, 1))
ax.grid(False)
guardar(fig, "nb01_bifurcacion.png")

**Tu turno.** Haz un acercamiento dentro de la ventana de periodo 3 (por ejemplo, $r$ entre 3.84 y
3.857, $x$ entre 0.44 y 0.54). ¿Qué ves? ¿Por qué se le llama autosimilar?

## 2. Sensibilidad a condiciones iniciales

In [ ]:
def serie(r, x0, n):
    out = np.empty(n)
    x = x0
    for i in range(n):
        out[i] = x
        x = r * x * (1 - x)
    return out

R_SENS, EPS = 3.9, 1e-10
a, b = serie(R_SENS, X0, 60), serie(R_SENS, X0 + EPS, 60)
paso = int(np.argmax(np.abs(a - b) > 0.1))
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(a, color=AZUL, label="x₀ = 0.4")
ax.plot(b, color=NARANJA, ls="--", label="x₀ = 0.4 + 10⁻¹⁰")
ax.axvline(paso, color=TINTA2, lw=0.8)
ax.set(xlabel="paso n", ylabel="x", title=f"r = {R_SENS}: se separan en el paso {paso}")
ax.legend(loc="lower left", ncols=2)
guardar(fig, "nb01_sensibilidad.png")
RESULTADOS["sensibilidad"] = {"r": R_SENS, "epsilon": EPS, "paso_de_separacion": paso}

## 3. Exponente de Lyapunov

$\lambda = \lim \frac{1}{N}\sum \ln|r(1-2x_n)|$. Si $\lambda > 0$, dos trayectorias cercanas se separan exponencialmente: caos.

In [ ]:
def lyapunov(r_min, r_max, n_r=4000, transitorio=1000, n=10000):
    r = xp.linspace(r_min, r_max, n_r)
    x = xp.full(n_r, X0)
    for _ in range(transitorio):
        x = r * x * (1 - x)
    s = xp.zeros(n_r)
    for _ in range(n):
        x = r * x * (1 - x)
        s += xp.log(xp.abs(r * (1 - 2 * x)) + 1e-300)
    return a_numpy(r), a_numpy(s / n)

r, lam = lyapunov(2.5, 4.0)
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(r, lam, color=TINTA, lw=0.6)
ax.axhline(0, color=TINTA2, lw=0.8)
ax.fill_between(r, 0, lam, where=lam > 0, color=NARANJA, alpha=0.35, label="λ > 0: caos")
ax.set(xlabel="r", ylabel="λ", ylim=(-2, 1), xlim=(2.5, 4))
ax.legend(loc="lower left")
guardar(fig, "nb01_lyapunov.png")
inicio = float(r[np.argmax(lam > 1e-3)])
print(f"El caos empieza en r ≈ {inicio:.4f};  λ(r = 4) = {lam[-1]:.4f}  (control: ln 2 = {np.log(2):.4f})")
RESULTADOS["lyapunov"] = {"inicio_caos_r": round(inicio, 4), "lyapunov_en_r4": round(float(lam[-1]), 4)}

## 4. La constante de Feigenbaum, en dos mapas distintos

Buscamos los valores **superestables** $R_n$: aquellos donde $x = 1/2$ (el máximo del mapa) pertenece a
una órbita de periodo $2^n$. El cociente $\delta_n = (R_{n-1}-R_{n-2})/(R_n-R_{n-1})$ tiende a
$\delta = 4.669201609\ldots$ Lo comprobamos con la logística y con $r\sin(\pi x)$: si sale el mismo número,
eso es la **universalidad**.

**Control:** el primer superestable de periodo 2 de la logística es exactamente $1 + \sqrt{5}$.

In [ ]:
DELTA = 4.669201609102990

def superestables(f, R0, rango_R1, niveles=9):
    def g(r, n):
        x = 0.5
        for _ in range(2 ** n):
            x = f(r, x)
        return x - 0.5
    def secante(n, a, b):
        fa, fb = g(a, n), g(b, n)
        for _ in range(200):
            if fb == fa:
                break
            c = b - fb * (b - a) / (fb - fa)
            a, fa, b, fb = b, fb, c, g(c, n)
            if abs(b - a) < 1e-15:
                break
        return b
    malla = np.linspace(*rango_R1, 20001)
    vals = [g(r, 1) for r in malla]
    k = next(i for i in range(1, len(malla)) if np.sign(vals[i]) != np.sign(vals[i - 1]))
    R = [R0, secante(1, malla[k - 1], malla[k])]
    for n in range(2, niveles):
        guia = R[-1] + (R[-1] - R[-2]) / DELTA
        R.append(secante(n, guia - 1e-6 * (R[-1] - R[-2]), guia))
    R = np.array(R)
    return R, (R[1:-1] - R[:-2]) / (R[2:] - R[1:-1])

R_log, d_log = superestables(lambda r, x: r * x * (1 - x), 2.0, (3.0, 3.4))
R_sen, d_sen = superestables(lambda r, x: r * np.sin(np.pi * x), 0.5, (0.65, 0.85))
print(f"Control: R1 = {R_log[1]:.10f}  vs  1 + √5 = {1 + 5 ** 0.5:.10f}")
print(f"δ logística, último nivel = {d_log[-1]:.6f};  δ seno = {d_sen[-1]:.6f};  referencia = {DELTA:.6f}")

n = np.arange(2, 2 + d_log.size)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(n, d_log, "o-", ms=4, color=AZUL, label="logística: r·x·(1−x)")
ax.plot(n, d_sen, "s--", ms=4, color=AQUA, label="seno: r·sin(πx)")
ax.axhline(DELTA, color=NARANJA, lw=1, label=f"δ = {DELTA:.9f}…")
ax.set(xlabel="nivel de duplicación n", ylabel="δₙ", ylim=(3.9, 5.0))
ax.legend()
guardar(fig, "nb01_feigenbaum.png")
RESULTADOS["feigenbaum"] = {"R1": float(R_log[1]), "R1_exacto": 1 + 5 ** 0.5,
                            "delta_logistica": [round(float(v), 6) for v in d_log],
                            "delta_seno": [round(float(v), 6) for v in d_sen], "delta_referencia": DELTA}
guardar_resultados("nb01_logistica.json")

## Tu turno

- Prueba otro mapa con un máximo cuadrático (por ejemplo $r\,x\,(1-x^2)$ ajustando el rango de $R_1$).
  ¿Sale la misma $\delta$? ¿Y con un máximo no cuadrático, como $r\,(1 - |2x-1|^{3})$?
- ¿Cuántas cifras de $\delta$ puedes afirmar? Anota también **lo que no funcionó**.
- Para llevar una figura al paper: `python simulaciones/copiar_al_paper.py nb01_feigenbaum.png`
  (o pídeselo a @simulador en Terminus).